# Notebook 00 — Setup & Dataset Download

**Project**: Landslide Identification Using Machine Learning  
**Phase**: Environment setup, dataset download, and train/test split  
**Dataset source**: http://gpcv.whu.edu.cn/data/landslide_dataset.html

---
Run this notebook **once** before any other notebook.

## 1. Install Dependencies

In [ ]:
# Run this cell only once
!pip install torch torchvision albumentations opencv-python-headless \
    scikit-learn pandas matplotlib seaborn openpyxl \
    hmmlearn torchinfo tqdm joblib scipy -q

## 2. Verify Imports

In [ ]:
import sys, os

# Add project root to path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, PROJECT_ROOT)

import torch, torchvision, albumentations, cv2, sklearn, pandas, numpy, hmmlearn
import matplotlib, seaborn

print(f'Python      : {sys.version}')
print(f'PyTorch     : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'OpenCV      : {cv2.__version__}')
print(f'hmmlearn    : {hmmlearn.__version__}')
print('All imports OK!')

## 3. (Colab Only) Mount Google Drive

In [ ]:
# Uncomment if running on Google Colab
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_ROOT = '/content/drive/MyDrive/Landslide_Project/project'
# sys.path.insert(0, os.path.dirname(PROJECT_ROOT))

print('Running locally — no Drive mount needed')

## 4. Dataset Download Instructions

In [ ]:
import config
from utils.data_utils import download_whu_dataset

download_whu_dataset(config.DATA_DIR)

### Manual download steps:
1. Visit: http://gpcv.whu.edu.cn/data/landslide_dataset.html
2. Download the **Landslide4Sense** dataset
3. Extract and copy:
   - Landslide images → `project/data/raw/landslide/`
   - Non-landslide images → `project/data/raw/non_landslide/`
4. Copy the Excel files from the parent folder:
   - `data set (1).xlsx` → `project/data/excel/`

## 5. Copy Excel Files

In [ ]:
import shutil

# Adjust source paths as needed
excel_sources = [
    os.path.join(PROJECT_ROOT, '..', 'data set (1).xlsx'),
    os.path.join(PROJECT_ROOT, '..', 'data set.xlsx'),
]

os.makedirs(config.EXCEL_DIR, exist_ok=True)
for src in excel_sources:
    src = os.path.abspath(src)
    if os.path.exists(src):
        dst = os.path.join(config.EXCEL_DIR, os.path.basename(src))
        shutil.copy2(src, dst)
        print(f'Copied: {os.path.basename(src)}')
    else:
        print(f'Not found: {src}')

## 6. Verify Dataset Counts

In [ ]:
from utils.data_utils import verify_dataset_counts

counts = verify_dataset_counts(config.RAW_DATA_DIR)
print(f"\nTotal images: {sum(counts.values())}")

## 7. Create Train / Test Split

In [ ]:
from utils.data_utils import split_dataset

split_dataset(config.RAW_DATA_DIR, config.PROCESSED_DATA_DIR, seed=config.RANDOM_SEED)
print('Split complete!')

## 8. Display Sample Images

In [ ]:
import matplotlib.pyplot as plt
import cv2
from pathlib import Path

def show_samples(folder, n=5, title=''):
    paths = list(Path(folder).glob('*.jpg'))[:n] + list(Path(folder).glob('*.png'))[:n]
    paths = paths[:n]
    if not paths:
        print(f'No images found in {folder}')
        return
    fig, axes = plt.subplots(1, len(paths), figsize=(4*len(paths), 4))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        img = cv2.imread(str(p))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(p.name[:20])
        ax.axis('off')
    plt.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

show_samples(os.path.join(config.PROCESSED_DATA_DIR, 'train', 'landslide'), n=5, title='Train — Landslide')
show_samples(os.path.join(config.PROCESSED_DATA_DIR, 'train', 'non_landslide'), n=5, title='Train — Non-Landslide')

## 9. Sanity Checks

Run these assertions to confirm dataset integrity before training.

In [ ]:
from phase1_alexnet.dataset import LandslideDataset, get_train_transforms

train_ds = LandslideDataset(config.PROCESSED_DATA_DIR, 'train', get_train_transforms())
test_ds  = LandslideDataset(config.PROCESSED_DATA_DIR, 'test',  get_train_transforms())

print(f'Train dataset size: {len(train_ds)}')
print(f'Test  dataset size: {len(test_ds)}')

img, label = train_ds[0]
print(f'Sample tensor shape: {img.shape}, label: {label} ({config.CLASS_NAMES[label]})')

labels = [train_ds.samples[i][1] for i in range(len(train_ds))]
print(f'Train landslide:     {labels.count(1)}')
print(f'Train non-landslide: {labels.count(0)}')

print('\nAll checks passed!')